<a href="https://colab.research.google.com/github/tomeravgil/Homework6CSCI6170/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

class GridWorldMDP:

    ACTIONS = {
        'U': (-1, 0),
        'D': (1, 0),
        'L': (0, -1),
        'R': (0, 1),
    }
    ACTION_LIST = ['U', 'D', 'L', 'R']
    ARROWS = {'U': '↑', 'D': '↓', 'L': '←', 'R': '→'}

    LEFT_OF = {'U': 'L', 'L': 'D', 'D': 'R', 'R': 'U'}
    RIGHT_OF = {'U': 'R', 'R': 'D', 'D': 'L', 'L': 'U'}

    def __init__(self, rows, cols, gamma=0.9, threshold=1e-6,
                 walls=None, terminals=None, step_reward=-0.04):
        self.rows = rows
        self.cols = cols
        self.gamma = gamma
        self.threshold = threshold
        self.step_reward = step_reward

        self.walls = walls or set()
        self.terminals = terminals or {}

        self.V = np.zeros((rows, cols))
        self.policy = np.full((rows, cols), ' ', dtype='U2')

    def in_bounds(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols

    def is_wall(self, r, c):
        return (r, c) in self.walls

    def is_terminal(self, r, c):
        return (r, c) in self.terminals

    def get_reward(self, r, c):
        if (r, c) in self.terminals:
            return self.terminals[(r, c)]
        return self.step_reward

    def attempt_move(self, r, c, action_name):
        """Try to move in a direction; bounce back if hitting wall/boundary."""
        dr, dc = self.ACTIONS[action_name]
        nr, nc = r + dr, c + dc
        if not self.in_bounds(nr, nc) or self.is_wall(nr, nc):
            return r, c
        return nr, nc

    def get_transitions(self, r, c, action_name):
        """Stochastic transitions: 80% intended, 10% each perpendicular."""
        results = []
        for prob, a in [(0.8, action_name),
                        (0.1, self.LEFT_OF[action_name]),
                        (0.1, self.RIGHT_OF[action_name])]:
            nr, nc = self.attempt_move(r, c, a)
            reward = self.get_reward(nr, nc)
            results.append((prob, nr, nc, reward))
        return results

    def train(self):
        """Run value iteration until convergence."""
        iterations = 0
        while True:
            delta = 0
            iterations += 1
            for r in range(self.rows):
                for c in range(self.cols):
                    if self.is_terminal(r, c) or self.is_wall(r, c):
                        continue
                    old_v = self.V[r, c]
                    action_values = []
                    for a in self.ACTION_LIST:
                        value = 0
                        for (prob, nr, nc, reward) in self.get_transitions(r, c, a):
                            value += prob * (reward + self.gamma * self.V[nr, nc])
                        action_values.append(value)
                    self.V[r, c] = max(action_values)
                    delta = max(delta, abs(old_v - self.V[r, c]))
            if delta < self.threshold:
                break

        # Set terminal values to their rewards
        for (r, c), reward in self.terminals.items():
            self.V[r, c] = reward

        self.extract_policy()
        print(f"Converged in {iterations} iterations")

    def extract_policy(self):
        """Derive the optimal policy from the converged value function."""
        for r in range(self.rows):
            for c in range(self.cols):
                if self.is_terminal(r, c) or self.is_wall(r, c):
                    continue
                best_a, best_val = None, float('-inf')
                for a in self.ACTION_LIST:
                    value = 0
                    for (prob, nr, nc, reward) in self.get_transitions(r, c, a):
                        value += prob * (reward + self.gamma * self.V[nr, nc])
                    if value > best_val:
                        best_val = value
                        best_a = a
                self.policy[r, c] = self.ARROWS[best_a]

    def display(self):
        """Print the value function and policy side by side."""
        print("\n=== Value Function ===")
        for r in range(self.rows):
            row_str = []
            for c in range(self.cols):
                if self.is_wall(r, c):
                    row_str.append("  ###  ")
                elif self.is_terminal(r, c):
                    row_str.append(f" [{self.terminals[(r,c)]:+.1f}] ")
                else:
                    row_str.append(f" {self.V[r,c]:+.3f}")

            print(" | ".join(row_str))

        print("\n=== Policy ===")
        for r in range(self.rows):
            row_str = []
            for c in range(self.cols):
                if self.is_wall(r, c):
                    row_str.append('#')
                elif self.is_terminal(r, c):
                    row_str.append('G' if self.terminals[(r,c)] > 0 else 'X')
                else:
                    row_str.append(self.policy[r, c])
            print("  ".join(row_str))


In [2]:

walls = {(1, 1)}
terminals = {(0, 3): +1.0, (1, 3): -1.0}

mdp = GridWorldMDP(rows=4, cols=4, gamma=0.9,
                   walls=walls, terminals=terminals, step_reward=-0.04)
mdp.train()
mdp.display()

print("\n\n========== DISCOUNT FACTOR EXPERIMENT ==========")
for g in [0.1, 0.5, 0.9, 0.99]:
    print(f"\n--- gamma = {g} ---")
    m = GridWorldMDP(rows=4, cols=4, gamma=g,
                     walls=walls, terminals=terminals, step_reward=-0.04)
    m.train()
    m.display()

Converged in 15 iterations

=== Value Function ===
 +0.610 |  +0.766 |  +0.928 |  [+1.0] 
 +0.487 |   ###   |  +0.585 |  [-1.0] 
 +0.373 |  +0.318 |  +0.427 |  +0.191
 +0.275 |  +0.241 |  +0.309 |  +0.219

=== Policy ===
→  →  →  G
↑  #  ←  X
↑  →  ↑  ↓
↑  ↑  ↑  ←


========== DISCOUNT FACTOR EXPERIMENT ==========

--- gamma = 0.1 ---
Converged in 6 iterations

=== Value Function ===
 -0.039 |  +0.024 |  +0.800 |  [+1.0] 
 -0.044 |   ###   |  -0.035 |  [-1.0] 
 -0.044 |  -0.044 |  -0.044 |  -0.044
 -0.044 |  -0.044 |  -0.044 |  -0.044

=== Policy ===
→  →  →  G
↑  #  ←  X
↑  →  ↑  ↓
↑  ↑  ↑  ←

--- gamma = 0.5 ---
Converged in 10 iterations

=== Value Function ===
 +0.097 |  +0.331 |  +0.845 |  [+1.0] 
 -0.001 |   ###   |  +0.213 |  [-1.0] 
 -0.044 |  -0.028 |  +0.040 |  -0.064
 -0.064 |  -0.056 |  -0.030 |  -0.058

=== Policy ===
→  →  →  G
↑  #  ↑  X
↑  →  ↑  ↓
↑  ↑  ↑  ←

--- gamma = 0.9 ---
Converged in 15 iterations

=== Value Function ===
 +0.610 |  +0.766 |  +0.928 |  [+1.0] 
 +

In [3]:
print("\n========== STEP REWARD EXPERIMENT ==========")
for sr in [-0.01, -0.04, -0.5]:
    print(f"\n--- step_reward = {sr} ---")
    m = GridWorldMDP(rows=4, cols=4, gamma=0.9,
                     walls=walls, terminals=terminals, step_reward=sr)
    m.train()
    m.display()


========== STEP REWARD EXPERIMENT ==========

--- step_reward = -0.01 ---
Converged in 17 iterations

=== Value Function ===
 +0.690 |  +0.812 |  +0.939 |  [+1.0] 
 +0.594 |   ###   |  +0.623 |  [-1.0] 
 +0.501 |  +0.426 |  +0.505 |  +0.317
 +0.422 |  +0.372 |  +0.419 |  +0.352

=== Policy ===
→  →  →  G
↑  #  ←  X
↑  →  ↑  ↓
↑  ↑  ↑  ←

--- step_reward = -0.04 ---
Converged in 15 iterations

=== Value Function ===
 +0.610 |  +0.766 |  +0.928 |  [+1.0] 
 +0.487 |   ###   |  +0.585 |  [-1.0] 
 +0.373 |  +0.318 |  +0.427 |  +0.191
 +0.275 |  +0.241 |  +0.309 |  +0.219

=== Policy ===
→  →  →  G
↑  #  ←  X
↑  →  ↑  ↓
↑  ↑  ↑  ←

--- step_reward = -0.5 ---
Converged in 14 iterations

=== Value Function ===
 -0.611 |  +0.066 |  +0.770 |  [+1.0] 
 -1.146 |   ###   |  +0.005 |  [-1.0] 
 -1.582 |  -1.277 |  -0.707 |  -1.059
 -1.970 |  -1.703 |  -1.299 |  -1.516

=== Policy ===
→  →  →  G
↑  #  ↑  X
↑  →  ↑  ←
↑  →  ↑  ↑


## Spaces, Reward Structure, and Discount Factor

The state space is 16 cells in a 4x4 grid, the action space is up, down, left, right. The rewards is +1 at the goal, -1 at the trap, and -0.04 per step. The discount factor is 0.9.

## Results

This class is an MDP since it satisfies the Markov property as the transition probabilities depend on only the current state and action, but not on how the agent arrived there. The stochastic transitions model realistic uncertainty in movement while the learned policy routes the agent towards the +1 goal and avoids the -1 trap. When the gamma is high, the agent values long term outcomes and takes safer paths, but when gamma is low, only states immediately next to rewards have a meaningful value, producing a mode shortsighted strategy.